# RepoLens Week 3 Documentation Pipeline

This notebook implements **Pathway A adapted for Markdown/RST**, using the same eight pinned documentation files selected in Weeks 1-2. It performs ingestion, inspection, an observed chunking comparison, validation, Part B export/lineage, an integrated Part C pipeline, and a second run for repeatability.

**Run all code cells in order. No model API key, training, embeddings or vector store is required.** All processing code uses the Python standard library; nothing from downloaded documentation is executed. The notebook embeds the implementation for independent Colab execution; the same readable implementation is committed in `src/week3_pipeline.py`.

**Execution evidence:** saved outputs were produced by a local Python/Jupyter kernel. They are not a claim of Colab execution. In Colab, select **Runtime > Run all**, inspect the output, save a copy of the executed notebook and download the final export ZIP. Runtime files are temporary.

Requirements: **FR2** needs documentation content; **FR3** needs citeable source identity; **RAI5** needs fixed repository/version provenance. This prepares their data inputs; it does not prove the later chatbot satisfies those requirements.

Network access is limited to fixed `raw.githubusercontent.com` URLs. Exact source and notice hashes are checked. No arbitrary crawling, external images, linked documents or include directives are followed.

## Cell 1 Workspace setup and fixed source metadata

Colab uses `/content/week3_project`; local execution uses a task-local `week3_project` folder. The original source manifest and license/notice identities are embedded so rerunning does not depend on a moving GitHub branch.

In [1]:
from pathlib import Path
import hashlib, json, platform, sys, urllib.request, zipfile
from datetime import datetime, timezone

IN_COLAB = "google.colab" in sys.modules
if not IN_COLAB:
    try:
        import google.colab
        IN_COLAB = True
    except ImportError:
        pass
BASE_DIR = Path("/content/week3_project") if IN_COLAB else Path.cwd() / "week3_project"
RAW_DIR = BASE_DIR / "data/raw"
PROCESSED_DIR = BASE_DIR / "data/processed"
for folder in [RAW_DIR, PROCESSED_DIR]:
    folder.mkdir(parents=True, exist_ok=True)
RAW, BASE, OUT = RAW_DIR, BASE_DIR, PROCESSED_DIR
MANIFEST = {'collection': 'RepoLens fixed initial documentation selection', 'verified_on': '2026-10-22', 'status': 'Source selection verified; ingestion not implemented; no upstream document copies committed', 'repositories': [{'repository': 'fastapi/fastapi', 'commit': '4bdfbe4513084b19e22421ce1f916d8c1ac462a0', 'license': 'MIT', 'license_url': 'https://raw.githubusercontent.com/fastapi/fastapi/4bdfbe4513084b19e22421ce1f916d8c1ac462a0/LICENSE', 'license_http_status': 200, 'redistribution': 'No upstream content redistributed in Week 1; review applicable conditions and file-specific exceptions before future copying.'}, {'repository': 'streamlit/streamlit', 'commit': 'f478c9783be8099f1d394d02dea77bd0697d93ea', 'license': 'Apache-2.0', 'license_url': 'https://raw.githubusercontent.com/streamlit/streamlit/f478c9783be8099f1d394d02dea77bd0697d93ea/LICENSE', 'license_http_status': 200, 'redistribution': 'No upstream content redistributed in Week 1; review applicable conditions and file-specific exceptions before future copying.'}, {'repository': 'scikit-learn/scikit-learn', 'commit': '718eedb7af8678255de080d4748d9f67e321ba41', 'license': 'BSD-3-Clause', 'license_url': 'https://raw.githubusercontent.com/scikit-learn/scikit-learn/718eedb7af8678255de080d4748d9f67e321ba41/COPYING', 'license_http_status': 200, 'redistribution': 'No upstream content redistributed in Week 1; review applicable conditions and file-specific exceptions before future copying.'}], 'documents': [{'id': 'D1', 'repository': 'fastapi/fastapi', 'commit': '4bdfbe4513084b19e22421ce1f916d8c1ac462a0', 'path': 'README.md', 'url': 'https://raw.githubusercontent.com/fastapi/fastapi/4bdfbe4513084b19e22421ce1f916d8c1ac462a0/README.md', 'browser_url': 'https://github.com/fastapi/fastapi/blob/4bdfbe4513084b19e22421ce1f916d8c1ac462a0/README.md', 'http_status': 200, 'sha256_utf8': '9b44160901909b924c7520227f5cdb44f7785560957b73987ff12379b3f82faa', 'license': 'MIT', 'relevance': 'Purpose, technologies, installation and running examples', 'collection_policy': 'Only this file; no linked assets, includes or external pages automatically followed'}, {'id': 'D2', 'repository': 'fastapi/fastapi', 'commit': '4bdfbe4513084b19e22421ce1f916d8c1ac462a0', 'path': 'docs/en/docs/tutorial/index.md', 'url': 'https://raw.githubusercontent.com/fastapi/fastapi/4bdfbe4513084b19e22421ce1f916d8c1ac462a0/docs/en/docs/tutorial/index.md', 'browser_url': 'https://github.com/fastapi/fastapi/blob/4bdfbe4513084b19e22421ce1f916d8c1ac462a0/docs/en/docs/tutorial/index.md', 'http_status': 200, 'sha256_utf8': '22d619c9497c8e88094fdba2b297c91ed29b21459b658a3586ae39a95fd65335', 'license': 'MIT', 'relevance': 'Installation and tutorial entry point', 'collection_policy': 'Only this file; no linked assets, includes or external pages automatically followed'}, {'id': 'D3', 'repository': 'fastapi/fastapi', 'commit': '4bdfbe4513084b19e22421ce1f916d8c1ac462a0', 'path': 'docs/en/docs/contributing.md', 'url': 'https://raw.githubusercontent.com/fastapi/fastapi/4bdfbe4513084b19e22421ce1f916d8c1ac462a0/docs/en/docs/contributing.md', 'browser_url': 'https://github.com/fastapi/fastapi/blob/4bdfbe4513084b19e22421ce1f916d8c1ac462a0/docs/en/docs/contributing.md', 'http_status': 200, 'sha256_utf8': '6dc18b05e194e864a9fd0353ae68bf89d66a531517306a38687f7b2d1b577126', 'license': 'MIT', 'relevance': 'Contribution entry point; external development guide is not ingested', 'collection_policy': 'Only this file; no linked assets, includes or external pages automatically followed'}, {'id': 'D4', 'repository': 'streamlit/streamlit', 'commit': 'f478c9783be8099f1d394d02dea77bd0697d93ea', 'path': 'README.md', 'url': 'https://raw.githubusercontent.com/streamlit/streamlit/f478c9783be8099f1d394d02dea77bd0697d93ea/README.md', 'browser_url': 'https://github.com/streamlit/streamlit/blob/f478c9783be8099f1d394d02dea77bd0697d93ea/README.md', 'http_status': 200, 'sha256_utf8': '45b271be0490145de6b0c245e2dc0ee5518b38f07eab5bc2419dd2c3923cd87e', 'license': 'Apache-2.0', 'relevance': 'Purpose, installation, running and contribution policy', 'collection_policy': 'Only this file; no linked assets, includes or external pages automatically followed'}, {'id': 'D5', 'repository': 'streamlit/streamlit', 'commit': 'f478c9783be8099f1d394d02dea77bd0697d93ea', 'path': 'CONTRIBUTING.md', 'url': 'https://raw.githubusercontent.com/streamlit/streamlit/f478c9783be8099f1d394d02dea77bd0697d93ea/CONTRIBUTING.md', 'browser_url': 'https://github.com/streamlit/streamlit/blob/f478c9783be8099f1d394d02dea77bd0697d93ea/CONTRIBUTING.md', 'http_status': 200, 'sha256_utf8': 'b101612cadc408d297d342aac0ab3be4a8ad9d016b6699ef4a257f1d44109a76', 'license': 'Apache-2.0', 'relevance': 'Contribution guidance and policy', 'collection_policy': 'Only this file; no linked assets, includes or external pages automatically followed'}, {'id': 'D6', 'repository': 'scikit-learn/scikit-learn', 'commit': '718eedb7af8678255de080d4748d9f67e321ba41', 'path': 'README.rst', 'url': 'https://raw.githubusercontent.com/scikit-learn/scikit-learn/718eedb7af8678255de080d4748d9f67e321ba41/README.rst', 'browser_url': 'https://github.com/scikit-learn/scikit-learn/blob/718eedb7af8678255de080d4748d9f67e321ba41/README.rst', 'http_status': 200, 'sha256_utf8': '3e0c151adde02aa2501477a7c2a17a87888abe58178c09a2a8eff794bbee3221', 'license': 'BSD-3-Clause', 'relevance': 'Purpose and project overview', 'collection_policy': 'Only this file; no linked assets, includes or external pages automatically followed'}, {'id': 'D7', 'repository': 'scikit-learn/scikit-learn', 'commit': '718eedb7af8678255de080d4748d9f67e321ba41', 'path': 'doc/install.rst', 'url': 'https://raw.githubusercontent.com/scikit-learn/scikit-learn/718eedb7af8678255de080d4748d9f67e321ba41/doc/install.rst', 'browser_url': 'https://github.com/scikit-learn/scikit-learn/blob/718eedb7af8678255de080d4748d9f67e321ba41/doc/install.rst', 'http_status': 200, 'sha256_utf8': '270cae08f287cd918f6ca9212c20b6b0984195e742369cb8c43136c524eff07d', 'license': 'BSD-3-Clause', 'relevance': 'Installation options and requirements', 'collection_policy': 'Only this file; no linked assets, includes or external pages automatically followed'}, {'id': 'D8', 'repository': 'scikit-learn/scikit-learn', 'commit': '718eedb7af8678255de080d4748d9f67e321ba41', 'path': 'doc/developers/contributing.rst', 'url': 'https://raw.githubusercontent.com/scikit-learn/scikit-learn/718eedb7af8678255de080d4748d9f67e321ba41/doc/developers/contributing.rst', 'browser_url': 'https://github.com/scikit-learn/scikit-learn/blob/718eedb7af8678255de080d4748d9f67e321ba41/doc/developers/contributing.rst', 'http_status': 200, 'sha256_utf8': 'e329bebf3986279ede0fc39dac46b36fd9e294c938ded19376481049824ac50d', 'license': 'BSD-3-Clause', 'relevance': 'Contribution guidance', 'collection_policy': 'Only this file; no linked assets, includes or external pages automatically followed'}]}
NOTICE_FILES = [{'relative_path': 'data/third_party/fastapi/LICENSE', 'url': 'https://raw.githubusercontent.com/fastapi/fastapi/4bdfbe4513084b19e22421ce1f916d8c1ac462a0/LICENSE', 'sha256': '4ec89ffc81485b97fec584b2d4a961032eeffe834453894fd9c1274906cc744e'}, {'relative_path': 'data/third_party/scikit-learn/COPYING', 'url': 'https://raw.githubusercontent.com/scikit-learn/scikit-learn/718eedb7af8678255de080d4748d9f67e321ba41/COPYING', 'sha256': '50d6a9d340f19ab355609917993114daf5f47e3161067bcf34955bbd05cd9cb0'}, {'relative_path': 'data/third_party/streamlit/LICENSE', 'url': 'https://raw.githubusercontent.com/streamlit/streamlit/f478c9783be8099f1d394d02dea77bd0697d93ea/LICENSE', 'sha256': 'c95bae1d1ce0235ecccd3560b772ec1efb97f348a79f0fbe0a634f0c2ccefe2c'}, {'relative_path': 'data/third_party/streamlit/NOTICES', 'url': 'https://raw.githubusercontent.com/streamlit/streamlit/f478c9783be8099f1d394d02dea77bd0697d93ea/NOTICES', 'sha256': '347b9560725ea8772029fdcb2735750b185edade727c556563b30b71bc2de1a8'}]
(BASE_DIR / "data/source_manifest.json").write_text(json.dumps(MANIFEST, indent=2), encoding="utf-8")
ATTRIBUTION_TEXT = "# RepoLens Week 3 Source Attribution and Modifications\n\nThe prepared JSONL files contain selected text from the eight source files listed in [source_manifest.json](source_manifest.json). They are derived documentation exports, not new documentation authored by RepoLens. Every chunk carries its original repository, commit, path, source URL, license, source hash and exact text-span identity. External image files, linked pages, source-code files and RST includes are not downloaded or incorporated by this pipeline.\n\n## Copyright and License Records\n\n- **FastAPI**: Copyright (c) 2018 Sebastián Ramírez. Original repository documentation is covered by the [MIT license retained here](third_party/fastapi/LICENSE). Original copyright/permission notice is preserved in full.\n- **Streamlit**: documentation from the Streamlit project, under [Apache License 2.0 retained here](third_party/streamlit/LICENSE). The distribution's complete [NOTICES file](third_party/streamlit/NOTICES) is retained unchanged, conservatively, even though the pipeline does not redistribute Streamlit's executable dependencies.\n- **scikit-learn**: Copyright (c) 2007-2026 The scikit-learn developers. All rights reserved. The [BSD 3-Clause license and disclaimer](third_party/scikit-learn/COPYING) are retained unchanged.\n\nThe pinned license URLs/commits and fetched notice-file hashes are recorded in [week3_rights_review.json](week3_rights_review.json). Repository trees were checked for root notice files; Streamlit's `NOTICES` is included. Source-specific text is kept with its provenance rather than being relabelled as original work. These attributions do not imply upstream endorsement.\n\n## Prominent Modification Notice\n\n**RepoLens modified the representation on 11 November 2026:** raw Markdown/RST text was decoded as UTF-8, CRLF/CR line endings were normalised to LF, then selected by paragraph/fenced-block spans and packaged into JSONL with added provenance fields. Text spans are otherwise exact; non-whitespace source content is retained. Heading labels are derived metadata, not upstream guarantees. No external assets or linked pages are fetched. The source bytes are downloaded into an ignored local raw folder, not committed here.\n\nBoth `data/processed/prepared_text.jsonl` and `data/pipeline_v1/documents/prepared_text.jsonl` are covered by this modification notice and the upstream licenses. Keep this attribution file and all applicable notices with copies of these exports. The original pinned URLs remain available for verification. No broad permission over unrelated linked material is claimed.\n"
RIGHTS_REVIEW = [{'repository': 'fastapi/fastapi', 'commit': '4bdfbe4513084b19e22421ce1f916d8c1ac462a0', 'license': 'MIT', 'license_url': 'https://raw.githubusercontent.com/fastapi/fastapi/4bdfbe4513084b19e22421ce1f916d8c1ac462a0/LICENSE', 'license_sha256': '4ec89ffc81485b97fec584b2d4a961032eeffe834453894fd9c1274906cc744e', 'notice_paths_in_tree': [], 'root_notices_copied': [], 'checked_on': '2026-11-11', 'review_scope': 'Pinned repository license, source file identity and root notices; no linked assets or external pages admitted', 'retained_files': [{'path': 'data/third_party/fastapi/LICENSE', 'sha256': '4ec89ffc81485b97fec584b2d4a961032eeffe834453894fd9c1274906cc744e'}]}, {'repository': 'streamlit/streamlit', 'commit': 'f478c9783be8099f1d394d02dea77bd0697d93ea', 'license': 'Apache-2.0', 'license_url': 'https://raw.githubusercontent.com/streamlit/streamlit/f478c9783be8099f1d394d02dea77bd0697d93ea/LICENSE', 'license_sha256': 'c95bae1d1ce0235ecccd3560b772ec1efb97f348a79f0fbe0a634f0c2ccefe2c', 'notice_paths_in_tree': ['NOTICES'], 'root_notices_copied': ['NOTICES'], 'checked_on': '2026-11-11', 'review_scope': 'Pinned repository license, source file identity and root notices; no linked assets or external pages admitted', 'retained_files': [{'path': 'data/third_party/streamlit/LICENSE', 'sha256': 'c95bae1d1ce0235ecccd3560b772ec1efb97f348a79f0fbe0a634f0c2ccefe2c'}, {'path': 'data/third_party/streamlit/NOTICES', 'sha256': '347b9560725ea8772029fdcb2735750b185edade727c556563b30b71bc2de1a8'}]}, {'repository': 'scikit-learn/scikit-learn', 'commit': '718eedb7af8678255de080d4748d9f67e321ba41', 'license': 'BSD-3-Clause', 'license_url': 'https://raw.githubusercontent.com/scikit-learn/scikit-learn/718eedb7af8678255de080d4748d9f67e321ba41/COPYING', 'license_sha256': '50d6a9d340f19ab355609917993114daf5f47e3161067bcf34955bbd05cd9cb0', 'notice_paths_in_tree': [], 'root_notices_copied': [], 'checked_on': '2026-11-11', 'review_scope': 'Pinned repository license, source file identity and root notices; no linked assets or external pages admitted', 'retained_files': [{'path': 'data/third_party/scikit-learn/COPYING', 'sha256': '50d6a9d340f19ab355609917993114daf5f47e3161067bcf34955bbd05cd9cb0'}]}]
(BASE_DIR / "data/ATTRIBUTION.md").write_text(ATTRIBUTION_TEXT, encoding="utf-8")
(BASE_DIR / "data/week3_rights_review.json").write_text(json.dumps(RIGHTS_REVIEW, indent=2), encoding="utf-8")
print("Execution environment:", "Google Colab" if IN_COLAB else "Local Jupyter kernel (not Colab)")
print("Python:", platform.python_version())
print("Workspace ready: data/raw and data/processed")

Execution environment: Google Colab
Python: 3.13.18
Workspace ready: data/raw and data/processed


## Source ingestion and Cell 3 file verification

Download exactly eight pinned inputs, preserving repository subfolders to avoid README-name collisions. Fail on missing files, redirects or changed bytes. This replaces the manual-upload step with a repeatable, allowlisted download; no dummy dataset is used. License/notice files are fetched separately for the derived exports, not ingested as Q&A documents.

In [2]:
def checked_download(url, expected_sha256, target):
    if not url.startswith("https://raw.githubusercontent.com/"):
        raise ValueError("Unexpected download host")
    request = urllib.request.Request(url, headers={"User-Agent": "RepoLens-Topic3/1.0"})
    with urllib.request.urlopen(request, timeout=45) as response:
        if response.geturl() != url:
            raise ValueError("Unexpected redirect")
        content = response.read(5_000_001)
    if len(content) > 5_000_000 or hashlib.sha256(content).hexdigest() != expected_sha256:
        raise ValueError("Source size or hash check failed")
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(content)

for source in MANIFEST["documents"]:
    target = RAW_DIR / source["repository"] / source["path"]
    checked_download(source["url"], source["sha256_utf8"], target)
    print(source["id"], str(target.relative_to(RAW_DIR)), "bytes:", target.stat().st_size)
for notice in NOTICE_FILES:
    checked_download(notice["url"], notice["sha256"], BASE_DIR / notice["relative_path"])
assert len(list(RAW_DIR.rglob("*.md"))) + len(list(RAW_DIR.rglob("*.rst"))) == 8
print("Upload/acquisition verified: exactly 8 project documents; license/notice files stored separately.")

D1 fastapi/fastapi/README.md bytes: 22228
D2 fastapi/fastapi/docs/en/docs/tutorial/index.md bytes: 7464
D3 fastapi/fastapi/docs/en/docs/contributing.md bytes: 271
D4 streamlit/streamlit/README.md bytes: 7008
D5 streamlit/streamlit/CONTRIBUTING.md bytes: 27548
D6 scikit-learn/scikit-learn/README.rst bytes: 7676
D7 scikit-learn/scikit-learn/doc/install.rst bytes: 14525
D8 scikit-learn/scikit-learn/doc/developers/contributing.rst bytes: 71320
Upload/acquisition verified: exactly 8 project documents; license/notice files stored separately.


## Cell 4 Common pipeline implementation

The code below is the committed implementation embedded as text, then loaded into a separate Python module namespace. It is local authored code, not downloaded repository code. This keeps the notebook self-contained. Review the definition cell or `src/week3_pipeline.py` to inspect every function. The SHA-256 recorded in each pipeline run identifies these exact implementation bytes.

In [3]:
import types
PIPELINE_CODE = '"""RepoLens Topic 3: a read-only, standard-library documentation pipeline.\n\nPublic source text is untrusted data. No source directives, code blocks, links or\nrepository instructions are executed. This prepares data, not embeddings or Q&A.\n"""\nfrom pathlib import Path\nfrom datetime import datetime, timezone\nfrom urllib.parse import urlparse\nimport argparse\nimport bisect\nimport hashlib\nimport json\nimport platform\nimport re\nimport statistics\nimport urllib.request\n\nPIPELINE_VERSION = \'repolens-week3-v1\'\nSUPPORTED = {\'.md\', \'.rst\', \'.txt\'}\nCONFIG = {\'target_characters\': 500, \'overlap\': 0,\n          \'cleaning\': \'UTF-8 strict; normalize CRLF/CR to LF; preserve indentation, links and directives as inert text\',\n          \'paragraph_policy\': \'Do not split atomic paragraphs or fenced blocks; 500 characters is a soft target\'}\n\n\ndef utc_now():\n    return datetime.now(timezone.utc).isoformat()\n\n\ndef sha256_bytes(value):\n    return hashlib.sha256(value).hexdigest()\n\n\ndef sha256_file(path):\n    return sha256_bytes(Path(path).read_bytes())\n\n\ndef write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(value, ensure_ascii=False, sort_keys=True, indent=2) + \'\\n\', encoding=\'utf-8\')\n\n\ndef read_jsonl(path):\n    return [json.loads(line) for line in Path(path).read_text(encoding=\'utf-8\').splitlines() if line.strip()]\n\n\ndef validate_manifest(manifest):\n    sources = manifest.get(\'documents\', [])\n    if len(sources) != 8 or len({s[\'id\'] for s in sources}) != 8:\n        raise ValueError(\'Expected exactly eight unique approved source records\')\n    repositories = {r[\'repository\']: r for r in manifest[\'repositories\']}\n    if set(repositories) != {\'fastapi/fastapi\', \'streamlit/streamlit\', \'scikit-learn/scikit-learn\'}:\n        raise ValueError(\'Unexpected repository allowlist\')\n    for source in sources:\n        repo = source[\'repository\']\n        if repo not in repositories or not re.fullmatch(r\'[0-9a-f]{40}\', source[\'commit\']):\n            raise ValueError(\'Unapproved repository or unpinned commit\')\n        relative = Path(source[\'path\'])\n        if relative.is_absolute() or \'..\' in relative.parts or relative.suffix.lower() not in SUPPORTED:\n            raise ValueError(\'Unsupported or unsafe source path\')\n        expected = f"https://raw.githubusercontent.com/{repo}/{source[\'commit\']}/{source[\'path\']}"\n        if source[\'url\'] != expected or urlparse(expected).hostname != \'raw.githubusercontent.com\':\n            raise ValueError(\'Source URL does not match approved identity\')\n        if not re.fullmatch(r\'[0-9a-f]{64}\', source[\'sha256_utf8\']):\n            raise ValueError(\'Missing expected source fingerprint\')\n        if source[\'commit\'] != repositories[repo][\'commit\'] or source[\'license\'] != repositories[repo][\'license\']:\n            raise ValueError(\'Inconsistent source commit or license metadata\')\n    return sorted(sources, key=lambda s: int(s[\'id\'][1:]))\n\n\ndef raw_path(base, source):\n    return Path(base) / \'data\' / \'raw\' / source[\'repository\'] / source[\'path\']\n\n\ndef download_sources(base, manifest, offline=False):\n    """Fetch only the eight allowlisted pinned files; verify before accepting."""\n    sources = validate_manifest(manifest)\n    report = []\n    for source in sources:\n        path = raw_path(base, source)\n        if not path.exists():\n            if offline:\n                raise FileNotFoundError(f\'Missing source: {path}\')\n            request = urllib.request.Request(source[\'url\'], headers={\'User-Agent\': \'RepoLens-Topic3/1.0\'})\n            with urllib.request.urlopen(request, timeout=45) as response:\n                if response.geturl() != source[\'url\']:\n                    raise ValueError(\'Unexpected source redirect; review before admitting\')\n                payload = response.read(5_000_001)\n            if len(payload) > 5_000_000:\n                raise ValueError(\'Source exceeds the V1 download limit\')\n            if sha256_bytes(payload) != source[\'sha256_utf8\']:\n                raise ValueError(f"Pinned source fingerprint mismatch: {source[\'id\']}")\n            path.parent.mkdir(parents=True, exist_ok=True)\n            path.write_bytes(payload)\n        if sha256_file(path) != source[\'sha256_utf8\']:\n            raise ValueError(f"Cached source fingerprint mismatch: {source[\'id\']}")\n        report.append({\'id\': source[\'id\'], \'path\': str(path.relative_to(base)),\n                       \'bytes\': path.stat().st_size, \'sha256\': sha256_file(path)})\n    return report\n\n\ndef load_text(path):\n    path = Path(path)\n    if path.suffix.lower() not in SUPPORTED:\n        raise ValueError(f\'Unsupported input type: {path.suffix}\')\n    text = path.read_bytes().decode(\'utf-8\', errors=\'strict\')\n    if \'\\x00\' in text or not text.strip():\n        raise ValueError(f\'Empty or binary-like document: {path}\')\n    # Do not flatten whitespace: indentation matters in commands, lists and RST.\n    return text.replace(\'\\r\\n\', \'\\n\').replace(\'\\r\', \'\\n\')\n\n\ndef headings(text, suffix):\n    lines = text.splitlines(keepends=True)\n    found = []\n    offset = 0\n    fenced = False\n    for i, line in enumerate(lines):\n        stripped = line.strip()\n        if suffix == \'.md\' and re.match(r\'^(`{3,}|~{3,})\', stripped):\n            fenced = not fenced\n        if not fenced:\n            md = re.match(r\'^#{1,6}\\s+(.+?)\\s*#*\\s*$\', stripped)\n            if suffix == \'.md\' and md:\n                found.append((offset, md.group(1)))\n            elif suffix == \'.rst\' and i + 1 < len(lines) and stripped:\n                underline = lines[i + 1].strip()\n                if len(underline) >= len(stripped) and re.fullmatch(r\'([=~^"`:+#*_-])\\1{2,}\', underline):\n                    found.append((offset, stripped))\n        offset += len(line)\n    return found\n\n\ndef atomic_blocks(text, markdown_fences=True):\n    """Return exact text spans; blank lines inside Markdown fences stay intact.\n\n    RST is treated as inert text. Paragraphs/indented blocks are kept whole;\n    this is not a full Markdown/RST parser or a semantic splitter.\n    """\n    lines = text.splitlines(keepends=True)\n    offsets = [0]\n    for line in lines:\n        offsets.append(offsets[-1] + len(line))\n    blocks = []\n    start = None\n    fence = None\n    for i, line in enumerate(lines):\n        stripped = line.strip()\n        marker = re.match(r\'^(`{3,}|~{3,})\', stripped) if markdown_fences else None\n        if start is None and stripped:\n            start = offsets[i]\n        if marker:\n            token = marker.group(1)\n            if fence is None:\n                fence = (token[0], len(token))\n            elif token[0] == fence[0] and len(token) >= fence[1]:\n                fence = None\n        if not stripped and fence is None and start is not None:\n            end = offsets[i]\n            if text[start:end].strip():\n                blocks.append((start, end))\n            start = None\n    if fence is not None:\n        raise ValueError(\'Unclosed Markdown fence; review extraction before preparing\')\n    if start is not None and text[start:].strip():\n        blocks.append((start, len(text)))\n    return blocks\n\n\ndef chunk_spans(text, method, size=500, markdown_fences=True):\n    if size <= 0:\n        raise ValueError(\'Chunk size must be positive\')\n    if method == \'fixed\':\n        return [(start, min(start + size, len(text))) for start in range(0, len(text), size)\n                if text[start:start + size].strip()]\n    if method != \'paragraph-aware\':\n        raise ValueError(\'Unknown chunk method\')\n    blocks = atomic_blocks(text, markdown_fences=markdown_fences)\n    chunks = []\n    current = None\n    for start, end in blocks:\n        if current is None:\n            current = [start, end]\n        elif end - current[0] <= size:\n            current[1] = end\n        else:\n            chunks.append(tuple(current))\n            current = [start, end]\n    if current:\n        chunks.append(tuple(current))\n    return chunks\n\n\ndef fence_intervals(text, markdown_fences=True):\n    if not markdown_fences:\n        return []\n    intervals = []\n    start = None\n    fence = None\n    offset = 0\n    for line in text.splitlines(keepends=True):\n        marker = re.match(r\'^\\s*(`{3,}|~{3,})\', line)\n        if marker:\n            token = marker.group(1)\n            if fence is None:\n                start, fence = offset, (token[0], len(token))\n            elif token[0] == fence[0] and len(token) >= fence[1]:\n                intervals.append((start, offset + len(line)))\n                fence = None\n        offset += len(line)\n    return intervals\n\n\ndef source_inspection(base, manifest):\n    reports = []\n    for source in validate_manifest(manifest):\n        path = raw_path(base, source)\n        text = load_text(path)\n        hs = headings(text, path.suffix.lower())\n        reports.append({\'id\': source[\'id\'], \'repository\': source[\'repository\'],\n                        \'relative_path\': str(path.relative_to(base)), \'bytes\': path.stat().st_size,\n                        \'characters\': len(text), \'lines\': len(text.splitlines()), \'heading_count\': len(hs),\n                        \'fenced_blocks\': len(fence_intervals(text, path.suffix.lower() == \'.md\')), \'non_ascii_characters\': sum(ord(c) > 127 for c in text),\n                        \'source_sha256\': sha256_file(path), \'preview\': text[:420],\n                        \'headings_preview\': [h[1] for h in hs[:6]],\n                        \'limitations\': \'Source markup retained; linked assets/includes not fetched; RST directives never executed.\'})\n    return reports\n\n\ndef compare_methods(base, manifest, size=500):\n    rows = []\n    for source in validate_manifest(manifest):\n        text = load_text(raw_path(base, source))\n        fences = fence_intervals(text, Path(source[\'path\']).suffix.lower() == \'.md\')\n        for method in [\'fixed\', \'paragraph-aware\']:\n            spans = chunk_spans(text, method, size, markdown_fences=Path(source[\'path\']).suffix.lower() == \'.md\')\n            lengths = [end - start for start, end in spans]\n            boundaries = [end for _, end in spans[:-1]]\n            rows.append({\'source_id\': source[\'id\'], \'method\': method, \'chunk_count\': len(spans),\n                         \'minimum_characters\': min(lengths), \'maximum_characters\': max(lengths),\n                         \'median_characters\': statistics.median(lengths),\n                         \'oversize_chunks\': sum(n > size for n in lengths),\n                         \'mid_line_boundaries\': sum(0 < end < len(text) and text[end - 1] != \'\\n\' and text[end] != \'\\n\' for end in boundaries),\n                         \'boundaries_inside_fences\': sum(any(a < end < b for a, b in fences) for end in boundaries),\n                         \'preview\': text[spans[0][0]:spans[0][1]][:420]})\n    totals = {}\n    for method in [\'fixed\', \'paragraph-aware\']:\n        subset = [r for r in rows if r[\'method\'] == method]\n        totals[method] = {key: sum(r[key] for r in subset) for key in\n                          [\'chunk_count\', \'oversize_chunks\', \'mid_line_boundaries\', \'boundaries_inside_fences\']}\n        totals[method][\'maximum_characters\'] = max(r[\'maximum_characters\'] for r in subset)\n    fixed, para = totals[\'fixed\'], totals[\'paragraph-aware\']\n    selected = (\'paragraph-aware\' if (para[\'boundaries_inside_fences\'], para[\'mid_line_boundaries\']) <\n                (fixed[\'boundaries_inside_fences\'], fixed[\'mid_line_boundaries\']) else \'fixed\')\n    return {\'configuration\': CONFIG, \'per_source\': rows, \'totals\': totals, \'selected_method\': selected,\n            \'decision_rule\': \'Prefer fewer boundaries inside fenced blocks, then fewer mid-line boundaries; otherwise retain simple fixed splitting.\',\n            \'scope_of_result\': \'Observable chunk-boundary comparison, not retrieval or answer-quality evaluation.\'}\n\n\ndef make_records(base, manifest, method, size=500):\n    records = []\n    for source in validate_manifest(manifest):\n        text = load_text(raw_path(base, source))\n        spans = chunk_spans(text, method, size, markdown_fences=Path(source[\'path\']).suffix.lower() == \'.md\')\n        line_offsets = [0] + [m.end() for m in re.finditer(\'\\n\', text)]\n        hs = headings(text, Path(source[\'path\']).suffix.lower())\n        for n, (start, end) in enumerate(spans, 1):\n            active = [title for offset, title in hs if offset <= start]\n            sections = ([active[-1]] if active else [\'Document introduction\']) + [title for offset, title in hs if start < offset < end]\n            sections = list(dict.fromkeys(sections))\n            content = text[start:end]\n            records.append({\'record_id\': f"{source[\'id\']}-{n:04d}", \'source_id\': source[\'id\'],\n                            \'text\': content, \'repository\': source[\'repository\'], \'document_path\': source[\'path\'],\n                            \'commit\': source[\'commit\'], \'source_url\': source[\'url\'], \'browser_url\': source[\'browser_url\'],\n                            \'source_sha256\': sha256_file(raw_path(base, source)), \'license\': source[\'license\'],\n                            \'sections\': sections, \'line_start\': bisect.bisect_right(line_offsets, start),\n                            \'line_end\': bisect.bisect_right(line_offsets, max(start, end - 1)),\n                            \'char_start\': start, \'char_end\': end, \'page\': None,\n                            \'processing\': {\'method\': method, \'target_characters\': size, \'overlap\': 0,\n                                           \'atomic_block_soft_limit\': method == \'paragraph-aware\'},\n                            \'text_sha256\': sha256_bytes(content.encode(\'utf-8\'))})\n    return records\n\n\ndef validate_records(records, base, manifest):\n    if not records:\n        raise ValueError(\'No prepared records\')\n    sources = {s[\'id\']: s for s in validate_manifest(manifest)}\n    ids = set()\n    by_source = {id: [] for id in sources}\n    texts = {id: load_text(raw_path(base, s)) for id, s in sources.items()}\n    for record in records:\n        id = record[\'source_id\']\n        if id not in sources or record[\'record_id\'] in ids or not record[\'text\'].strip():\n            raise ValueError(\'Invalid identity, duplicate ID or empty text\')\n        ids.add(record[\'record_id\'])\n        source = sources[id]\n        for key in [\'repository\', \'commit\', \'license\']:\n            if record[key] != source[key]:\n                raise ValueError(f\'Wrong {key} for {id}\')\n        if record[\'document_path\'] != source[\'path\'] or record[\'source_url\'] != source[\'url\'] or record[\'source_sha256\'] != source[\'sha256_utf8\']:\n            raise ValueError(\'Invalid source provenance\')\n        if record[\'text\'] != texts[id][record[\'char_start\']:record[\'char_end\']]:\n            raise ValueError(\'Prepared text is not an exact source span\')\n        if record[\'text_sha256\'] != sha256_bytes(record[\'text\'].encode(\'utf-8\')):\n            raise ValueError(\'Prepared text hash mismatch\')\n        if not record[\'sections\'] or record[\'line_start\'] < 1 or record[\'line_end\'] < record[\'line_start\']:\n            raise ValueError(\'Missing section/line provenance\')\n        by_source[id].append(record)\n    # Every non-whitespace source character must survive. Whitespace-only gaps are allowed.\n    for id, rows in by_source.items():\n        if not rows:\n            raise ValueError(f\'No records for {id}\')\n        rows.sort(key=lambda x: x[\'char_start\'])\n        previous = 0\n        for row in rows:\n            if row[\'char_start\'] < previous or texts[id][previous:row[\'char_start\']].strip():\n                raise ValueError(\'Overlapping chunks or non-whitespace data loss\')\n            previous = row[\'char_end\']\n        if texts[id][previous:].strip():\n            raise ValueError(\'Source tail lost\')\n    return {\'status\': \'PASS\', \'record_count\': len(records), \'source_count\': len(by_source),\n            \'unique_record_ids\': len(ids), \'non_whitespace_content_preserved\': True,\n            \'exact_source_spans\': True, \'provenance_verified\': True}\n\n\ndef export_records(path, records):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open(\'w\', encoding=\'utf-8\', newline=\'\\n\') as handle:\n        for record in records:\n            handle.write(json.dumps(record, ensure_ascii=False, sort_keys=True) + \'\\n\')\n    exported = read_jsonl(path)\n    if exported != records or not path.stat().st_size:\n        raise ValueError(\'Export round-trip failed\')\n\n\ndef lineage_records(base, manifest, records, output, method):\n    rows = []\n    for source in validate_manifest(manifest):\n        selected = [r[\'record_id\'] for r in records if r[\'source_id\'] == source[\'id\']]\n        rows.append({\'source_id\': source[\'id\'], \'source_filename\': str(raw_path(base, source).relative_to(base)),\n                     \'source_url\': source[\'url\'], \'source_sha256\': sha256_file(raw_path(base, source)),\n                     \'source_version\': source[\'commit\'], \'repository\': source[\'repository\'],\n                     \'access_rights\': source[\'license\'] + \'; retained notices in data/third_party; linked assets excluded\',\n                     \'prepared_output\': str(Path(output).relative_to(base)), \'prepared_sha256\': sha256_file(output),\n                     \'record_count\': len(selected), \'first_record_id\': selected[0], \'last_record_id\': selected[-1],\n                     \'processing_note\': method + \'; \' + CONFIG[\'cleaning\'],\n                     \'known_limitations\': \'Simple text/heading handling; no execution, include expansion, semantic retrieval or answer evaluation.\'})\n    return rows\n\n\ndef run_part_b(base, manifest):\n    base = Path(base)\n    comparison = compare_methods(base, manifest)\n    selected = comparison[\'selected_method\']\n    records = make_records(base, manifest, selected)\n    validation = validate_records(records, base, manifest)\n    output = base / \'data/processed/prepared_text.jsonl\'\n    export_records(output, records)\n    write_json(base / \'data/processed/chunking_comparison.json\', comparison)\n    write_json(base / \'data/processed/inspection.json\', source_inspection(base, manifest))\n    write_json(base / \'data/week3_manifest.json\', {\n        \'pipeline_version\': PIPELINE_VERSION, \'generated_at_utc\': utc_now(),\n        \'purpose\': \'Part B preparation lineage; all eight sources map into one combined JSONL with stable source IDs\',\n        \'configuration\': {**CONFIG, \'method\': selected},\n        \'records\': lineage_records(base, manifest, records, output, selected), \'validation\': validation})\n    print(f\'PART B PASS | sources={validation["source_count"]} | chunks={len(records)} | method={selected}\')\n    return comparison, validation\n\n\ndef execute_pipeline(base, manifest, configuration, implementation_sha256):\n    base = Path(base)\n    if not re.fullmatch(\'[0-9a-f]{64}\', implementation_sha256):\n        raise ValueError(\'Record the actual implementation SHA-256\')\n    # Recheck raw bytes at ingestion, including offline reruns; never silently refresh.\n    ingestion = download_sources(base, manifest, offline=True)\n    inspection = source_inspection(base, manifest)\n    records = make_records(base, manifest, configuration[\'method\'], configuration[\'target_characters\'])\n    validation = validate_records(records, base, manifest)\n    output = base / \'data/pipeline_v1/documents/prepared_text.jsonl\'\n    export_records(output, records)\n    validation.update({\'output_count\': 1, \'output_bytes\': output.stat().st_size,\n                       \'output_sha256\': sha256_file(output)})\n    run = {\'pipeline_version\': PIPELINE_VERSION, \'generated_at_utc\': utc_now(),\n           \'execution_environment\': {\'python\': platform.python_version(), \'system\': platform.system(),\n                                     \'colab\': \'google.colab\' in __import__(\'sys\').modules},\n           \'implementation_sha256\': implementation_sha256, \'data_type\': \'documents\',\n           \'configuration\': configuration, \'sources\': ingestion, \'inspection\': inspection,\n           \'outputs\': [str(output.relative_to(base))],\n           \'source_output_mapping\': lineage_records(base, manifest, records, output, configuration[\'method\']),\n           \'validation\': validation}\n    write_json(output.parent / \'pipeline_manifest.json\', run)\n    print(f\'SUCCESS: documents | Inputs: {len(ingestion)} | Records: {len(records)} | Outputs: 1\')\n    return run\n\n\ndef repeatability_check(base, manifest, configuration, implementation_sha256):\n    first = execute_pipeline(base, manifest, configuration, implementation_sha256)\n    second = execute_pipeline(base, manifest, configuration, implementation_sha256)\n    keys = [\'sources\', \'configuration\', \'validation\', \'source_output_mapping\', \'implementation_sha256\']\n    for key in keys:\n        if first[key] != second[key]:\n            raise AssertionError(f\'Repeatability failure: {key}\')\n    part_b = Path(base) / \'data/processed/prepared_text.jsonl\'\n    part_c = Path(base) / second[\'outputs\'][0]\n    if part_b.read_bytes() != part_c.read_bytes():\n        raise AssertionError(\'Part B and C outputs differ\')\n    report = {\'status\': \'PASS\', \'checks\': keys, \'run_count\': 2,\n              \'first_generated_at_utc\': first[\'generated_at_utc\'], \'second_generated_at_utc\': second[\'generated_at_utc\'],\n              \'part_b_matches_part_c\': True, \'prepared_sha256\': sha256_file(part_c),\n              \'sources\': len(second[\'sources\']), \'records\': second[\'validation\'][\'record_count\'],\n              \'environment\': second[\'execution_environment\'],\n              \'limitation\': \'Same-environment repeatability smoke check; not cross-environment reproducibility or answer-quality proof.\'}\n    write_json(part_c.parent / \'repeatability_report.json\', report)\n    print(\'PASS: source fingerprints, prepared bytes, counts, mappings and configuration match; Part B equals Part C.\')\n    return report\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--base\', type=Path, default=Path(__file__).resolve().parents[1])\n    parser.add_argument(\'--offline\', action=\'store_true\')\n    args = parser.parse_args()\n    base = args.base.resolve()\n    manifest = json.loads((base / \'data/source_manifest.json\').read_text(encoding=\'utf-8\'))\n    download_sources(base, manifest, offline=args.offline)\n    comparison, _ = run_part_b(base, manifest)\n    repeatability_check(base, manifest, {**CONFIG, \'method\': comparison[\'selected_method\']}, sha256_file(__file__))\n\n\nif __name__ == \'__main__\':\n    main()\n'
CODE_SHA256 = hashlib.sha256(PIPELINE_CODE.encode("utf-8")).hexdigest()
P = types.ModuleType("repolens_week3_notebook_pipeline")
exec(compile(PIPELINE_CODE, "week3_pipeline.py", "exec"), P.__dict__)
assert len(P.validate_manifest(MANIFEST)) == 8
print("Common source-selection, validation, export and lineage utilities ready.")
print("Implementation SHA-256:", CODE_SHA256)

Common source-selection, validation, export and lineage utilities ready.
Implementation SHA-256: 5ec0c744f9c8e120c302392b08d3ecf64fe9fcc83d03940eeeacf1382f9d53d4


## Cell 4A and Part A inspection

Inspect all eight sources (exceeds the two-file/subset checkpoint). Markdown and RST are read as UTF-8 text. Newline normalization is allowed; indentation, code blocks, URLs and inert directives remain. The simple heading recogniser is not a full Markdown/RST parser. A source hash proves identity, not legal permission or data quality.

In [4]:
lineage_entries = {}
INSPECTION = P.source_inspection(BASE_DIR, MANIFEST)
for item in INSPECTION:
    print(item["id"], item["repository"], "chars:", item["characters"], "lines:", item["lines"],
          "headings:", item["heading_count"], "fences:", item["fenced_blocks"])
for source_id in ["D1", "D4"]:
    item = next(x for x in INSPECTION if x["id"] == source_id)
    print("\nRepresentative actual source:", source_id, item["relative_path"])
    print(item["preview"])
print("Lineage registry initialised; inspection completed.")

D1 fastapi/fastapi chars: 22011 lines: 533 headings: 29 fences: 12
D2 fastapi/fastapi chars: 7458 lines: 141 headings: 5 fences: 3
D3 fastapi/fastapi chars: 271 lines: 7 headings: 2 fences: 0
D4 streamlit/streamlit chars: 6965 lines: 150 headings: 12 fences: 4
D5 streamlit/streamlit chars: 27536 lines: 594 headings: 54 fences: 31
D6 scikit-learn/scikit-learn chars: 7676 lines: 217 headings: 19 fences: 0
D7 scikit-learn/scikit-learn chars: 14505 lines: 426 headings: 15 fences: 0
D8 scikit-learn/scikit-learn chars: 71305 lines: 1665 headings: 30 fences: 0

Representative actual source: D1 data/raw/fastapi/fastapi/README.md
<p align="center">
  <a href="https://fastapi.tiangolo.com"><img src="https://fastapi.tiangolo.com/img/logo-margin/logo-teal.png" alt="FastAPI"></a>
</p>
<p align="center">
    <em>FastAPI framework, high performance, easy to learn, fast to code, ready for production</em>
</p>
<p align="center">
<a href="https://github.com/fastapi/fastapi/actions?query=workflow%3ATest+

## Cell 5A1 Compare fixed-size and paragraph-aware chunks

Both methods use the same source text and a 500-character target. The adapted paragraph-aware method keeps atomic paragraphs and Markdown fenced code blocks intact; its limit is **soft**. RST underline headings are not interpreted as Markdown fences. Compare actual boundary counts, chunk counts and maximum lengths before selecting a method. This is structural preparation evidence, not a claim about semantic retrieval or answer accuracy.

In [5]:
COMPARISON = P.compare_methods(BASE_DIR, MANIFEST, size=500)
for method, totals in COMPARISON["totals"].items():
    print(method, json.dumps(totals, sort_keys=True))
for source_id in ["D1", "D4"]:
    for row in COMPARISON["per_source"]:
        if row["source_id"] == source_id:
            print("\n", source_id, row["method"], "sample:")
            print(row["preview"])
CHOSEN_METHOD = COMPARISON["selected_method"]
print("\nSelected method:", CHOSEN_METHOD)
print("Decision rule:", COMPARISON["decision_rule"])

fixed {"boundaries_inside_fences": 17, "chunk_count": 320, "maximum_characters": 500, "mid_line_boundaries": 299, "oversize_chunks": 0}
paragraph-aware {"boundaries_inside_fences": 0, "chunk_count": 350, "maximum_characters": 2749, "mid_line_boundaries": 0, "oversize_chunks": 49}

 D1 fixed sample:
<p align="center">
  <a href="https://fastapi.tiangolo.com"><img src="https://fastapi.tiangolo.com/img/logo-margin/logo-teal.png" alt="FastAPI"></a>
</p>
<p align="center">
    <em>FastAPI framework, high performance, easy to learn, fast to code, ready for production</em>
</p>
<p align="center">
<a href="https://github.com/fastapi/fastapi/actions?query=workflow%3ATest+event%3Apush+branch%3Amaster">
    <img src="http

 D1 paragraph-aware sample:
<p align="center">
  <a href="https://fastapi.tiangolo.com"><img src="https://fastapi.tiangolo.com/img/logo-margin/logo-teal.png" alt="FastAPI"></a>
</p>
<p align="center">
    <em>FastAPI framework, high performance, easy to learn, fast to code, rea

## Cell 5A2 Export prepared document records

Every record carries the source ID, repository, original path/URL, pinned commit, source hash, section labels, original normalized-text line range, exact character span, method and content hash. `page` is null because these are Markdown/RST inputs, not PDFs. Do not invent page numbers. The eight sources are combined into one export; source-specific record IDs preserve the many-to-one relationship.

In [6]:
RECORDS = P.make_records(BASE_DIR, MANIFEST, CHOSEN_METHOD, size=500)
prepared_path = PROCESSED_DIR / "prepared_text.jsonl"
P.export_records(prepared_path, RECORDS)
print("Prepared export:", str(prepared_path.relative_to(BASE_DIR)))
print("Records:", len(RECORDS))
example = dict(RECORDS[0])
example["text"] = example["text"][:200] + " [preview only]"
print(json.dumps(example, ensure_ascii=False, indent=2))

Prepared export: data/processed/prepared_text.jsonl
Records: 350
{
  "record_id": "D1-0001",
  "source_id": "D1",
  "text": "<p align=\"center\">\n  <a href=\"https://fastapi.tiangolo.com\"><img src=\"https://fastapi.tiangolo.com/img/logo-margin/logo-teal.png\" alt=\"FastAPI\"></a>\n</p>\n<p align=\"center\">\n    <em>FastAPI framework,  [preview only]",
  "repository": "fastapi/fastapi",
  "document_path": "README.md",
  "commit": "4bdfbe4513084b19e22421ce1f916d8c1ac462a0",
  "source_url": "https://raw.githubusercontent.com/fastapi/fastapi/4bdfbe4513084b19e22421ce1f916d8c1ac462a0/README.md",
  "browser_url": "https://github.com/fastapi/fastapi/blob/4bdfbe4513084b19e22421ce1f916d8c1ac462a0/README.md",
  "source_sha256": "9b44160901909b924c7520227f5cdb44f7785560957b73987ff12379b3f82faa",
  "license": "MIT",
  "sections": [
    "Document introduction"
  ],
  "line_start": 1,
  "line_end": 20,
  "char_start": 0,
  "char_end": 1050,
  "page": null,
  "processing": {
    "method": "paragrap

## B3 Validation and observed trade-off

Check more than file existence: JSON round-trip, unique IDs, exact source spans, complete source identities and preservation of every non-whitespace source character. Inspect representative records too. Keeping larger blocks avoids command/paragraph cuts but produces some chunks above 500 characters; later retrieval must consider a token/context budget.

In [7]:
exported = P.read_jsonl(prepared_path)
assert exported == RECORDS
VALIDATION = P.validate_records(exported, BASE_DIR, MANIFEST)
print(json.dumps(VALIDATION, indent=2))
print("Export bytes:", prepared_path.stat().st_size)
print("Oversize paragraph-aware chunks:", COMPARISON["totals"]["paragraph-aware"]["oversize_chunks"])
assert VALIDATION["source_count"] == 8
print("B3 VALIDATION: PASS")

{
  "status": "PASS",
  "record_count": 350,
  "source_count": 8,
  "unique_record_ids": 350,
  "non_whitespace_content_preserved": true,
  "exact_source_spans": true,
  "provenance_verified": true
}
Export bytes: 480029
Oversize paragraph-aware chunks: 49
B3 VALIDATION: PASS


## B4 registration and Cell 6A Part B lineage

Record one source-to-combined-export relationship per source with a stable record range. Save actual version/license metadata rather than leaving the starter's `Not verified` placeholders. Retain source licenses and Streamlit NOTICES beside the export; see the repository's `data/ATTRIBUTION.md` for scope and modifications.

In [8]:
lineage_entries = P.lineage_records(BASE_DIR, MANIFEST, RECORDS, prepared_path, CHOSEN_METHOD)
P.write_json(PROCESSED_DIR / "inspection.json", INSPECTION)
P.write_json(PROCESSED_DIR / "chunking_comparison.json", COMPARISON)
P.write_json(BASE_DIR / "data/week3_manifest.json", {
    "pipeline_version": P.PIPELINE_VERSION, "generated_at_utc": P.utc_now(),
    "purpose": "Part B preparation lineage; all eight sources map into one combined JSONL with stable source IDs",
    "configuration": {**P.CONFIG, "method": CHOSEN_METHOD},
    "records": lineage_entries, "validation": VALIDATION})
print("COMMON LINEAGE MANIFEST GENERATED:", len(lineage_entries), "relationships")

COMMON LINEAGE MANIFEST GENERATED: 8 relationships


## Cell 6B Verify saved Part B lineage

In [9]:
saved = json.loads((BASE_DIR / "data/week3_manifest.json").read_text())
assert len(saved["records"]) == 8
for row in saved["records"]:
    assert P.sha256_file(BASE_DIR / row["source_filename"]) == row["source_sha256"]
    assert P.sha256_file(BASE_DIR / row["prepared_output"]) == row["prepared_sha256"]
    assert row["record_count"] > 0
print("COMMON LINEAGE VALIDATION: PASS; 8 source/output mappings verified")

COMMON LINEAGE VALIDATION: PASS; 8 source/output mappings verified


## Cell 7 Integrated pipeline configuration

Reuse the Part B source set, preparation method and size. This is a finite **batch ETL** workflow for a prepared-data destination: acquire/land raw sources, extract, transform, validate and load the prepared JSONL; retain the raw landing copy for provenance. No scheduled refresh or streaming service is needed.

In [10]:
PIPELINE_CONFIGURATION = {**P.CONFIG, "method": CHOSEN_METHOD}
print("Cell 7 ready: documents pipeline only")
print(json.dumps(PIPELINE_CONFIGURATION, indent=2))

Cell 7 ready: documents pipeline only
{
  "target_characters": 500,
  "overlap": 0,
  "cleaning": "UTF-8 strict; normalize CRLF/CR to LF; preserve indentation, links and directives as inert text",
  "paragraph_policy": "Do not split atomic paragraphs or fenced blocks; 500 characters is a soft target",
  "method": "paragraph-aware"
}


## Cell 8A Run the integrated document pipeline

Re-ingest the checked source bytes, inspect, prepare/store, validate and record lineage. Sources are read-only; no text is executed.

In [11]:
FIRST_RUN = P.execute_pipeline(BASE_DIR, MANIFEST, PIPELINE_CONFIGURATION, CODE_SHA256)
print("Pipeline manifest: data/pipeline_v1/documents/pipeline_manifest.json")
print("Actual execution environment:", FIRST_RUN["execution_environment"])

SUCCESS: documents | Inputs: 8 | Records: 350 | Outputs: 1
Pipeline manifest: data/pipeline_v1/documents/pipeline_manifest.json
Actual execution environment: {'python': '3.13.18', 'system': 'Linux', 'colab': True}


## Cell 9 Rerun and verify repeatability

Run the same pipeline a second time. Compare fingerprints, output bytes/counts, configuration and source-to-output mappings; timestamps may differ. Also compare the final Part C export with Part B. This is a same-environment smoke test, not full cross-environment reproducibility or a quality/accuracy benchmark.

In [12]:
SECOND_RUN = P.execute_pipeline(BASE_DIR, MANIFEST, PIPELINE_CONFIGURATION, CODE_SHA256)
CHECK_KEYS = ["sources", "configuration", "validation", "source_output_mapping", "implementation_sha256"]
for key in CHECK_KEYS:
    assert FIRST_RUN[key] == SECOND_RUN[key], "Repeatability failed: " + key
part_c = BASE_DIR / SECOND_RUN["outputs"][0]
assert prepared_path.read_bytes() == part_c.read_bytes()
REPORT = {
    "status": "PASS", "checks": CHECK_KEYS, "run_count": 2,
    "first_generated_at_utc": FIRST_RUN["generated_at_utc"],
    "second_generated_at_utc": SECOND_RUN["generated_at_utc"],
    "part_b_matches_part_c": True, "prepared_sha256": P.sha256_file(part_c),
    "sources": len(SECOND_RUN["sources"]), "records": SECOND_RUN["validation"]["record_count"],
    "environment": SECOND_RUN["execution_environment"],
    "limitation": "Same-environment repeatability smoke check; not cross-environment reproducibility or answer-quality proof."}
P.write_json(part_c.parent / "repeatability_report.json", REPORT)
print("PASS: source fingerprints, prepared bytes, counts, mappings and configuration match.")
print("PASS: Part B and Part C exports are byte-identical.")
print(json.dumps(REPORT, indent=2))

SUCCESS: documents | Inputs: 8 | Records: 350 | Outputs: 1
PASS: source fingerprints, prepared bytes, counts, mappings and configuration match.
PASS: Part B and Part C exports are byte-identical.
{
  "status": "PASS",
  "checks": [
    "sources",
    "configuration",
    "validation",
    "source_output_mapping",
    "implementation_sha256"
  ],
  "run_count": 2,
  "first_generated_at_utc": "2026-11-11T05:05:51.388925+00:00",
  "second_generated_at_utc": "2026-11-11T05:05:51.549487+00:00",
  "part_b_matches_part_c": true,
  "prepared_sha256": "9a71abd2df1a530c6b58d06a41ccf329b23132310421a7766a661ad020834b66",
  "sources": 8,
  "records": 350,
  "environment": {
    "python": "3.13.18",
    "system": "Linux",
    "colab": true
  },
  "limitation": "Same-environment repeatability smoke check; not cross-environment reproducibility or answer-quality proof."
}


## Save the permitted exports before the runtime resets

This creates a ZIP of Part B exports/lineage, Part C outputs/manifests and retained licenses/notices, excluding raw inputs and secret/runtime files. Save the notebook itself separately using Colab's download/save options. The inventory, diagram and decisions are committed in the repository's `docs/` folder. To download in Colab, uncomment the final two lines. No browser action is claimed by the saved local outputs.

In [13]:
EXPORT_ZIP = BASE_DIR / "week3_data_exports.zip"
allowed = [BASE_DIR / "data/processed", BASE_DIR / "data/pipeline_v1", BASE_DIR / "data/third_party"]
with zipfile.ZipFile(EXPORT_ZIP, "w", zipfile.ZIP_DEFLATED) as archive:
    for folder in allowed:
        for path in sorted(folder.rglob("*")):
            if path.is_file():
                archive.write(path, path.relative_to(BASE_DIR))
    for relative in ["data/week3_manifest.json", "data/source_manifest.json", "data/ATTRIBUTION.md", "data/week3_rights_review.json"]:
        archive.write(BASE_DIR / relative, relative)
print("Export ZIP ready:", EXPORT_ZIP.name, "bytes:", EXPORT_ZIP.stat().st_size)
# In Google Colab, uncomment to download:
# from google.colab import files
# files.download(str(EXPORT_ZIP))

Export ZIP ready: week3_data_exports.zip bytes: 290156


## Completion and limits

Check that every code cell ran successfully, then inspect the recorded samples, comparison, validation and repeatability outputs. The generated source/output hashes establish identity within this run. They do not prove copyright permissions, semantic correctness, fairness, downstream retrieval quality or answer accuracy.

Known limitations: soft chunk limit, simple headings, retained HTML/RST markup, unexpanded includes, no linked assets, no live refresh and no semantic model evaluation. Source content may contain instructions: those remain inert data. The week-three work expands only source preparation/knowledge inputs in the Week 2 conceptual architecture.

After a Colab rerun, save the actually executed notebook and exports to the existing repository. A local run is already available for verification; do not relabel its saved outputs as Colab evidence.